# 🌏 Bit-MC-SSM: 多言語ベースモデル 事前準備ノートブック (Colab / 無料枠)

**目的**: 日本語(Wikipedia + くだけたWeb文体)＋英語のバイリンガル・トークナイザと、事前学習用トークン列(`data_bin`)をここ(無料Colab)で作り、RunPod等の課金GPU環境には**300M本体モデルの学習だけ**を持ち込む。

この段階(トークナイザ訓練・データ前処理)はCPU/ネットワーク律速でGPUを必要としないため、課金環境の時間を消費しないようにColab側で完結させる。

⚠️ **ランタイムはCPUのみに設定すること** (`ランタイム > ランタイムのタイプを変更 > ハードウェアアクセラレータ: なし`)。このノートブックはGPUを一切使わないため、GPUランタイムで実行すると無料枠のGPU時間を無駄に消費するうえ、GPUランタイムは需要が高く切断/プリエンプションされやすいので不安定になりやすい。

⚠️ **作業はColabのローカルディスク(`/content/`)で行い、各段階が成功したことを確認してから明示的にGoogle Driveへコピーする。** `drive.mount()`で見えるパスは実体がFUSE経由の非同期書き込みで、ローカルの書き込み(`os.rename`等)がその場で成功したように見えても、実際にDrive側へアップロードされる前にランタイムが切断されるとデータが失われることがある。ローカルで完成 → 検証 → コピー、の順にすることでこれを避ける。

### 手順
1. リポジトリ取得 & 依存インストール
2. Google Drive マウント (最終成果物の保存先。作業自体はローカルディスクで行う)
3. 多言語(日本語Wikipedia + CC-100口語Web + 英語SmolLM)バイトレベルBPEトークナイザ訓練 → 検証 → Driveへバックアップ
4. 圧縮率チェック(語彙サイズが日本語に対して妥当か検証)
5. `ja_en_mix` データセットで事前学習用バイナリを生成 → 検証 → Driveへバックアップ

## 1. リポジトリ取得 & 依存ライブラリ

In [ ]:
!git clone https://github.com/fukayatti/BitMC-SSM.git
%cd BitMC-SSM
!pip install -q datasets tokenizers tqdm

## 2. Google Drive マウント & 作業ディレクトリ設定
`LOCAL_ROOT`(ローカルディスク、高速だがランタイム終了で消える)で実際の処理を行い、
`DRIVE_ROOT`(Google Drive、永続化用)には各段階の完了後に明示的にコピーする。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/bitmc_ssm'   # 永続化先 (RunPodへ転送する対象)
LOCAL_ROOT = '/content/bitmc_ssm'                  # 高速なローカル作業ディレクトリ (ランタイム終了で消える)

TOKENIZER_DIR = f'{LOCAL_ROOT}/tokenizer'
DATA_BIN = f'{LOCAL_ROOT}/data/train_tokens.bin'

import os
os.makedirs(TOKENIZER_DIR, exist_ok=True)
os.makedirs(f'{LOCAL_ROOT}/data', exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/tokenizer', exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/data', exist_ok=True)

## 3. 多言語トークナイザ訓練
3系統(日本語Wikipedia=正式文体 / 日本語CC-100=くだけたWeb文体 / 英語SmolLM)から学習することで、
キーボード入力(IME予測)に必要な口語日本語も、翻訳に必要な英語も、両方うまく圧縮できる語彙を作る。

`vocab_size=49152` は「小さすぎて日本語がバイト単位まで分解される」のと「edgeモデルとして語彙埋め込みが重くなりすぎる」のバランスを取った値。次のセルで実際の圧縮率を検証する。

⚠️ **まずは小さいドキュメント数(下記デフォルト)でパイプライン全体が通ることを確認してから、本番規模に上げること。** 各ソースはストリーミング中に`{TOKENIZER_DIR}/_corpus_cache/`(ローカルディスク)へ逐次テキストとしてキャッシュされ、2,000件ごとに進捗ログが出るので、途中で切断されても同一ランタイム内でなら再ダウンロードなしで再開できる(進捗が見えず数分固まったように見えるのは仕様ではない)。`!`セルはエラーが出ても自動で次に進んでしまうため、次のアサーションセルで必ず成功を確認すること。

⚠️ ドキュメント数や`max_doc_chars`を後で変える場合は、**先に`_corpus_cache/`内の対応する`.txt`を削除**すること(キャッシュがあるとそちらが優先され、変更が反映されない)。

⚠️ 文書数を増やしすぎると、BPEトレーナーの`Count pairs`ステップでRAMを使い切って途中停止することがある(特に長文なSmolLM cosmopedia-v2)。`--max_doc_chars`で1文書あたりの文字数に上限をかけてメモリ使用量を抑えている。

In [ ]:
# 動作確認用の小規模設定。問題なければ ja_wiki_docs=8000 / ja_web_docs=8000 / en_docs=16000 程度に上げる
# (49kの語彙を学習するのにWikipedia全体規模は不要。各ソースは{TOKENIZER_DIR}/_corpus_cache/にキャッシュされるので再実行は高速)。
# max_doc_chars は1文書あたりの文字数上限(長い文書でRAMを使い切るのを防ぐ)。
!python python/train_tokenizer.py \
    --vocab_size 49152 \
    --ja_wiki_docs 2000 \
    --ja_web_docs 2000 \
    --en_docs 2000 \
    --max_doc_chars 4000 \
    --out_dir {TOKENIZER_DIR}

In [ ]:
# `!` セルはエラーで停止しないため、ここで明示的にファイルが生成されたか確認する(まだローカルディスク上)。
import os

vocab_path = f'{TOKENIZER_DIR}/vocab.json'
merges_path = f'{TOKENIZER_DIR}/merges.txt'

assert os.path.exists(vocab_path) and os.path.exists(merges_path), (
    f"❌ トークナイザ訓練が失敗しています(vocab.json/merges.txtが{TOKENIZER_DIR}に見つかりません)。\n"
    "上のセルの出力をスクロールして、エラーメッセージ(ネットワークエラー/HFレート制限など)がないか確認してください。"
)
print(f"✅ Tokenizer files confirmed locally at {TOKENIZER_DIR}")

In [ ]:
# ローカルで検証済みのファイルだけをGoogle Driveへコピー(バックアップ)する。
!cp -v {TOKENIZER_DIR}/vocab.json {TOKENIZER_DIR}/merges.txt {TOKENIZER_DIR}/tokenizer_config.json {DRIVE_ROOT}/tokenizer/

## 4. 圧縮率チェック(日本語1文字あたりトークン数)
1.0に近いほど非効率(≒バイト単位分解)。0.4〜0.6程度なら実用的な圧縮ができている目安。
悪ければ`vocab_size`や`ja_wiki_docs`/`ja_web_docs`を増やして再訓練する。

In [ ]:
from tokenizers import ByteLevelBPETokenizer

tok = ByteLevelBPETokenizer.from_file(f'{TOKENIZER_DIR}/vocab.json', f'{TOKENIZER_DIR}/merges.txt')

samples = [
    "今日はいい天気ですね、散歩にでも行こうかな。",
    "えー、まじで？それはウケるｗｗｗ",
    "明日の会議の資料をまだ作成していないので、急いで準備する必要がある。",
]

for s in samples:
    n_tokens = len(tok.encode(s).ids)
    ratio = n_tokens / len(s)
    print(f"chars={len(s):3d}  tokens={n_tokens:3d}  tokens/char={ratio:.2f}  | {s}")

## 5. 事前学習用トークン列の生成 (`ja_en_mix`)
英語50% / 日本語Wikipedia25% / 日本語CC-100(口語)25% で重み付きラウンドロビン混合。
`--num_samples`は目標トークン数から逆算して調整する(まずは小さめで動作確認→本番は大きく)。

In [ ]:
# 動作確認用の小規模設定。問題なければ num_samples を目標トークン数から逆算して増やす。
!python python/preprocess_data.py \
    --dataset ja_en_mix \
    --tokenizer_dir {TOKENIZER_DIR} \
    --num_samples 500 \
    --out {DATA_BIN}

In [ ]:
import os

assert os.path.exists(DATA_BIN), (
    f"❌ 前処理が失敗しています({DATA_BIN}が見つかりません)。上のセルの出力を確認してください。"
)
print(f"✅ {DATA_BIN} ({os.path.getsize(DATA_BIN) / (1024*1024):.2f} MB) confirmed locally")

In [ ]:
# ローカルで検証済みの data_bin だけをGoogle Driveへコピー(バックアップ)する。
!cp -v {DATA_BIN} {DRIVE_ROOT}/data/

## ✅ 完了 — 次のステップ
`Google Drive: bitmc_ssm/tokenizer/` と `bitmc_ssm/data/train_tokens.bin` をRunPodへ転送し、
`python/train.py --vocab_size 49152 --data_bin train_tokens.bin ...` で300M本体モデルの学習を開始する。